# 🦅 PREDATOR Analytics v63.0-ELITE: K8s Kaggle Node
**Architecture**: K3s + Helm + GitOps + zrok

Цей блокнот розгортає:
1. **K3s (Kubernetes)** - полегшена версія для Kaggle.
2. **Helm** - пакетний менеджер для деплою чартів.
3. **zrok** - тунель для доступу до API / UI.
4. **PREDATOR ELITE** - повний стек сервісів згідно з `values-kaggle.yaml`.

In [ ]:
# 1. Встановлення системних компонентів
!curl -sfL https://get.k3s.io | sh -s - --write-kubeconfig-mode 644 --docker=false
!curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash

# 2. Налаштування Kubeconfig
import os
os.environ['KUBECONFIG'] = '/etc/rancher/k3s/k3s.yaml'

# 3. Завантаження Helm чартів PREDATOR
!git clone https://github.com/dima1203oleg/predator-analytics.git repo

# 4. Розгортання через Helm
%cd repo/deploy/helm/predator
!helm install predator . -f values.yaml -f values-kaggle.yaml --namespace predator-elite --create-namespace

# 5. Запуск zrok для експозиції API
!wget -q https://github.com/openziti/zrok/releases/download/v0.4.42/zrok_0.4.42_linux_amd64.tar.gz && tar -xzf zrok_*.tar.gz && chmod +x zrok
!./zrok enable 1eeje4um7yvA

import subprocess
import threading

def run_tunnel():
    # Прокидаємо порт 8000 (Core API) з K8s сервісу
    # Спочатку робимо Port-forward
    subprocess.Popen(["kubectl", "port-forward", "-n", "predator-elite", "svc/predator-core-api", "8000:8000"])
    # Потім запускаємо zrok
    p = subprocess.Popen(["./zrok", "share", "public", "http://localhost:8000", "--headless"], stdout=subprocess.PIPE, text=True)
    for line in p.stdout:
        if "access your share at" in line:
            print(f"\n🚀 PREDATOR K8s Node is LIVE!")
            print(f"🔗 URL: {line.split('access your share at')[-1].strip()}")
            break

threading.Thread(target=run_tunnel, daemon=True).start()

# 6. Моніторинг статусів
!kubectl get pods -n predator-elite